# LeetCode 744 — Find Smallest Letter Greater Than Target bằng Python

**Link đề:** https://leetcode.com/problems/find-smallest-letter-greater-than-target/
**Độ khó:** Easy · **Chủ đề:** Array, Binary Search

```python
class Solution:
    def nextGreatestLetter(self, letters: list[str], target: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu so sánh chuỗi/ký tự trong Python (`'a' < 'c'`, `ord`, `chr`) và yêu cầu "**lớn hơn hẳn**" (strictly greater).
2. Giải trực giác bằng duyệt tuyến tính `O(n)`, rồi tận dụng dữ kiện **đã sắp xếp** để dùng **Binary Search** `O(log n)`.
3. Nắm chắc mẫu binary search "**tìm vị trí đầu tiên thỏa điều kiện**" (upper bound) — bất biến, điều kiện dừng, tránh lặp vô hạn.
4. Biết dùng thư viện chuẩn `bisect.bisect_right` và xử lý **quay vòng** (wrap-around) bằng `% n`.
5. Trace, chứng minh, test đối chiếu và luyện biến thể (lower bound, đếm phần tử, tìm khoảng).

## 1. Đề bài

Cho mảng ký tự `letters` **đã sắp xếp không giảm** (non-decreasing, có thể lặp lại) và ký tự `target`. Mảng có ít nhất 2 ký tự khác nhau.

Trả về **ký tự nhỏ nhất trong `letters` lớn hơn hẳn `target`** (theo thứ tự từ điển). Nếu không có ký tự nào như vậy, trả về `letters[0]` (quay vòng về đầu).

| letters | target | Output | Vì sao |
|---|:---:|:---:|---|
| `["c","f","j"]` | `"a"` | `"c"` | `c` là ký tự nhỏ nhất > `a` |
| `["c","f","j"]` | `"c"` | `"f"` | phải **lớn hơn hẳn**, nên bỏ qua chính `c` |
| `["x","x","y","y"]` | `"z"` | `"x"` | không ký tự nào > `z` → `letters[0]` |

**Constraints:** `2 <= len(letters) <= 10^4`; mọi ký tự là chữ thường tiếng Anh; `letters` sắp xếp không giảm; có ≥ 2 ký tự khác nhau; `target` là chữ thường.

## 2. Nền tảng Python

### 2.1. So sánh ký tự

Python so sánh chuỗi theo **mã Unicode** từng ký tự. Với chữ thường `a..z`, mã tăng dần nên `'a' < 'b' < ... < 'z'` — đúng thứ tự từ điển. `ord(ch)` cho mã số, `chr(code)` đổi ngược lại.

In [1]:
print('a' < 'c', 'c' < 'c', 'c' <= 'c', 'z' > 'y')
print(ord('a'), ord('c'), ord('z'))
print(chr(97), chr(ord('c') + 1))

True False True True
97 99 122
a d


### 2.2. "Lớn hơn hẳn" khác "lớn hơn hoặc bằng"

Đề dùng `>` (strictly greater). Với `target = 'c'` và `letters` chứa `'c'`, đáp án **không** phải `'c'`. Đây là chỗ sai phổ biến nhất của bài.

In [2]:
letters = ["c", "f", "j"]
target = "c"
print("ký tự > target :", [ch for ch in letters if ch > target])
print("ký tự >= target:", [ch for ch in letters if ch >= target])

ký tự > target : ['f', 'j']
ký tự >= target: ['c', 'f', 'j']


### 2.3. Chia lấy nguyên `//` và chỉ số giữa

Binary search cần điểm giữa của đoạn `[lo, hi)`: `mid = (lo + hi) // 2`. Phép `//` làm tròn **xuống**, nên `mid` luôn `< hi` khi `lo < hi` — tính chất quan trọng để vòng lặp không kẹt.

In [3]:
for lo, hi in [(0, 10), (0, 1), (3, 5), (4, 5)]:
    mid = (lo + hi) // 2
    print(f"lo={lo}, hi={hi} -> mid={mid}  (lo <= mid < hi: {lo <= mid < hi})")

lo=0, hi=10 -> mid=5  (lo <= mid < hi: True)
lo=0, hi=1 -> mid=0  (lo <= mid < hi: True)
lo=3, hi=5 -> mid=4  (lo <= mid < hi: True)
lo=4, hi=5 -> mid=4  (lo <= mid < hi: True)


### 2.4. Phép `%` để quay vòng

Nếu vị trí tìm được là `n` (vượt cuối mảng), "quay về đầu" là `n % n = 0`. Còn vị trí `i < n` thì `i % n = i` không đổi. Một biểu thức xử lý cả hai trường hợp.

In [4]:
n = 4
for i in [0, 2, 3, 4]:
    print(f"{i} % {n} = {i % n}")

0 % 4 = 0
2 % 4 = 2
3 % 4 = 3
4 % 4 = 0


## 3. Cách 1 — Duyệt tuyến tính

Vì `letters` đã sắp xếp, ký tự **đầu tiên** (từ trái) lớn hơn `target` chính là ký tự **nhỏ nhất** lớn hơn `target`. Duyệt từ trái, gặp là trả. Không gặp → `letters[0]`.

In [5]:
def next_letter_linear(letters: list[str], target: str) -> str:
    for ch in letters:
        if ch > target:
            return ch
    return letters[0]

print(next_letter_linear(["c", "f", "j"], "a"))
print(next_letter_linear(["c", "f", "j"], "c"))
print(next_letter_linear(["x", "x", "y", "y"], "z"))

c
f
x


### Cách 1b — Theo gợi ý của LeetCode: thử 26 chữ cái

Hint của đề: "*thử xem từng chữ cái tiếp theo trong 26 chữ có nằm trong mảng không*". Bỏ `letters` vào `set`, rồi thử `target+1, target+2, ...` (quay vòng sau `z`). Tối đa 25 lần thử.

In [6]:
def next_letter_alphabet(letters: list[str], target: str) -> str:
    present = set(letters)
    code = ord(target)
    for step in range(1, 27):
        ch = chr(ord('a') + (code - ord('a') + step) % 26)
        if ch in present and ch > target:
            return ch
    return letters[0]

print(next_letter_alphabet(["c", "f", "j"], "a"))
print(next_letter_alphabet(["c", "f", "j"], "c"))
print(next_letter_alphabet(["x", "x", "y", "y"], "z"))

c
f
x


Điều kiện `ch > target` cần thiết vì sau khi quay vòng qua `z` ta gặp các chữ **nhỏ hơn** `target` — những chữ đó không hợp lệ; nếu không có chữ nào lớn hơn, ta trả `letters[0]` theo đề.

**Chi phí:** Cách 1 là `O(n)`; Cách 1b là `O(n)` để dựng set + `O(26)`.

## 4. Nút thắt và Insight

Cách 1 **không dùng** dữ kiện "đã sắp xếp" một cách triệt để: nó vẫn có thể phải đọc gần hết `n = 10^4` phần tử (khi `target` lớn).

**Quan sát:** với mảng sắp xếp, điều kiện `letters[i] > target` có dạng

```text
i      : 0     1     2     3     4     5
letters: c     c     f     f     j     m        target = 'f'
> 'f' ?: False False False False True  True
```

Một dãy **False... rồi True...** — gọi là tính **đơn điệu** (monotonic). Ta cần tìm **vị trí True đầu tiên**. Với dãy đơn điệu, mỗi lần nhìn phần tử giữa ta loại được **một nửa**: đó là **Binary Search** — `O(log n)` (với `n = 10^4` chỉ khoảng 14 bước).

In [7]:
letters = ["c", "c", "f", "f", "j", "m"]
target = "f"
print([ch > target for ch in letters])

[False, False, False, False, True, True]


## 5. Cách 2 — Binary Search tự viết (lời giải chính)

Ta duy trì nửa khoảng `[lo, hi)` chứa **chắc chắn** vị trí True đầu tiên (hoặc `n` nếu không có True).

- Khởi tạo `lo = 0`, `hi = n` (cho phép kết quả `n` = "không có").
- Khi `lo < hi`: lấy `mid = (lo + hi) // 2`.
  - Nếu `letters[mid] > target` (True): đáp án ở `mid` **hoặc bên trái** → `hi = mid` (giữ `mid`).
  - Ngược lại (False): đáp án chắc chắn **bên phải** `mid` → `lo = mid + 1` (bỏ `mid`).
- Khi `lo == hi`: đó là vị trí True đầu tiên. Trả `letters[lo % n]`.

In [8]:
def next_letter_binary(letters: list[str], target: str) -> str:
    n = len(letters)
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if letters[mid] > target:
            hi = mid
        else:
            lo = mid + 1
    return letters[lo % n]

print(next_letter_binary(["c", "f", "j"], "a"))
print(next_letter_binary(["c", "f", "j"], "c"))
print(next_letter_binary(["x", "x", "y", "y"], "z"))

c
f
x


### Vì sao `hi = mid` mà không phải `hi = mid - 1`?

Khi `letters[mid] > target`, chính `mid` **có thể** là đáp án (True đầu tiên). Viết `mid - 1` sẽ vứt mất nó.

### Vì sao `lo = mid + 1`?

Khi `letters[mid] <= target`, `mid` là False → chắc chắn không phải đáp án, và mọi vị trí bên trái cũng False (đơn điệu). Bỏ hẳn `mid`.

### Vì sao vòng lặp luôn dừng?

`mid < hi` (do `//` làm tròn xuống), nên `hi = mid` làm `hi` giảm; `lo = mid + 1 > lo` làm `lo` tăng. Mỗi vòng độ dài `hi - lo` giảm ít nhất 1 → chắc chắn kết thúc.

## 6. Trace từng bước

`letters = ["c","c","f","f","j","m"]`, `target = "f"`. Dãy điều kiện: `F F F F T T` → đáp án vị trí 4 (`'j'`).

| Vòng | lo | hi | mid | letters[mid] | > 'f'? | Cập nhật |
|---:|---:|---:|---:|:---:|:---:|---|
| 1 | 0 | 6 | 3 | f | ❌ | lo = 4 |
| 2 | 4 | 6 | 5 | m | ✅ | hi = 5 |
| 3 | 4 | 5 | 4 | j | ✅ | hi = 4 |
| — | 4 | 4 | | | | dừng → `letters[4 % 6] = 'j'` |

In [9]:
def trace_next_letter(letters, target):
    n = len(letters)
    lo, hi = 0, n
    k = 0
    print(f"letters = {letters}, target = {target!r}")
    while lo < hi:
        k += 1
        mid = (lo + hi) // 2
        cond = letters[mid] > target
        old = (lo, hi)
        if cond:
            hi = mid
        else:
            lo = mid + 1
        print(f"vòng {k}: lo={old[0]}, hi={old[1]}, mid={mid}, letters[mid]={letters[mid]!r}, "
              f"> target? {cond!s:<5} -> lo={lo}, hi={hi}")
    print(f"dừng: lo = hi = {lo} -> trả letters[{lo} % {n}] = {letters[lo % n]!r}")
    return letters[lo % n]

trace_next_letter(["c", "c", "f", "f", "j", "m"], "f")

letters = ['c', 'c', 'f', 'f', 'j', 'm'], target = 'f'
vòng 1: lo=0, hi=6, mid=3, letters[mid]='f', > target? False -> lo=4, hi=6
vòng 2: lo=4, hi=6, mid=5, letters[mid]='m', > target? True  -> lo=4, hi=5
vòng 3: lo=4, hi=5, mid=4, letters[mid]='j', > target? True  -> lo=4, hi=4
dừng: lo = hi = 4 -> trả letters[4 % 6] = 'j'


'j'

### Trace trường hợp quay vòng

`["x","x","y","y"]`, `target = "z"`: mọi vị trí đều False → `lo` chạy tới `n = 4` → `4 % 4 = 0` → `'x'`.

In [10]:
trace_next_letter(["x", "x", "y", "y"], "z")
print()
trace_next_letter(["c", "f", "j"], "a")

letters = ['x', 'x', 'y', 'y'], target = 'z'
vòng 1: lo=0, hi=4, mid=2, letters[mid]='y', > target? False -> lo=3, hi=4
vòng 2: lo=3, hi=4, mid=3, letters[mid]='y', > target? False -> lo=4, hi=4
dừng: lo = hi = 4 -> trả letters[4 % 4] = 'x'

letters = ['c', 'f', 'j'], target = 'a'
vòng 1: lo=0, hi=3, mid=1, letters[mid]='f', > target? True  -> lo=0, hi=1
vòng 2: lo=0, hi=1, mid=0, letters[mid]='c', > target? True  -> lo=0, hi=0
dừng: lo = hi = 0 -> trả letters[0 % 3] = 'c'


'c'

## 7. Cách 3 — `bisect.bisect_right` của thư viện chuẩn

Module `bisect` cài sẵn binary search trên list đã sắp xếp:
- `bisect_left(a, x)`: vị trí đầu tiên có `a[i] >= x` (chèn `x` vào **trước** các phần tử bằng `x`).
- `bisect_right(a, x)`: vị trí đầu tiên có `a[i] > x` (chèn **sau** các phần tử bằng `x`).

Bài này cần "đầu tiên **lớn hơn hẳn**" → đúng là `bisect_right`.

In [11]:
from bisect import bisect_left, bisect_right

a = ["c", "c", "f", "f", "j", "m"]
for x in ["a", "c", "f", "g", "m", "z"]:
    print(f"x={x!r}: bisect_left={bisect_left(a, x)}, bisect_right={bisect_right(a, x)}")

x='a': bisect_left=0, bisect_right=0
x='c': bisect_left=0, bisect_right=2
x='f': bisect_left=2, bisect_right=4
x='g': bisect_left=4, bisect_right=4
x='m': bisect_left=5, bisect_right=6
x='z': bisect_left=6, bisect_right=6


In [12]:
def next_letter_bisect(letters: list[str], target: str) -> str:
    i = bisect_right(letters, target)
    return letters[i % len(letters)]

print(next_letter_bisect(["c", "f", "j"], "a"))
print(next_letter_bisect(["c", "f", "j"], "c"))
print(next_letter_bisect(["x", "x", "y", "y"], "z"))

c
f
x


## 8. Code nộp LeetCode

### Binary search tự viết (khuyến nghị để hiểu và đi phỏng vấn)

In [13]:
class Solution:
    def nextGreatestLetter(self, letters: list[str], target: str) -> str:
        lo, hi = 0, len(letters)
        while lo < hi:
            mid = (lo + hi) // 2
            if letters[mid] > target:
                hi = mid
            else:
                lo = mid + 1
        return letters[lo % len(letters)]


sol = Solution()
print(sol.nextGreatestLetter(["c", "f", "j"], "a"))
print(sol.nextGreatestLetter(["c", "f", "j"], "c"))
print(sol.nextGreatestLetter(["x", "x", "y", "y"], "z"))

c
f
x


### Bản `bisect` và bản tuyến tính

Trên LeetCode đổi tên class thành `Solution`.

In [14]:
import bisect


class SolutionBisect:
    def nextGreatestLetter(self, letters: list[str], target: str) -> str:
        return letters[bisect.bisect_right(letters, target) % len(letters)]


class SolutionLinear:
    def nextGreatestLetter(self, letters: list[str], target: str) -> str:
        return next((ch for ch in letters if ch > target), letters[0])


print(SolutionBisect().nextGreatestLetter(["c", "f", "j"], "c"), SolutionLinear().nextGreatestLetter(["c", "f", "j"], "c"))

f f


`next(generator, default)` lấy phần tử **đầu tiên** của generator; nếu generator rỗng thì trả `default`. Đây là cách viết một dòng cho "tìm phần tử đầu tiên thỏa điều kiện, không có thì mặc định".

## 9. Vì sao đúng?

Gọi `P(i)` là mệnh đề `letters[i] > target`. Vì `letters` không giảm, `P` **đơn điệu**: nếu `P(i)` đúng thì `P(j)` đúng với mọi `j > i`. Gọi `ans` = chỉ số nhỏ nhất có `P` đúng (hoặc `n` nếu không có).

**Invariant:** `lo <= ans <= hi`; mọi `i < lo` có `P(i)` sai; mọi `i >= hi` (và `i < n`) có `P(i)` đúng.
- *Khởi đầu:* `lo = 0`, `hi = n` → hiển nhiên.
- *Nhánh `P(mid)` đúng:* `ans <= mid` → gán `hi = mid` giữ invariant.
- *Nhánh `P(mid)` sai:* do đơn điệu, mọi `i <= mid` sai → `ans >= mid + 1` → gán `lo = mid + 1` giữ invariant.
- *Kết thúc:* `lo == hi` ⇒ `ans = lo`.

Nếu `ans < n`, `letters[ans]` là phần tử **đầu tiên** > `target` trong mảng sắp xếp ⇒ là ký tự **nhỏ nhất** > `target`. Nếu `ans = n`, không có ký tự nào > `target` ⇒ đề yêu cầu `letters[0]` = `letters[n % n]`.

## 10. Độ phức tạp

Gọi `n = len(letters)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Duyệt tuyến tính | `O(n)` | `O(1)` |
| 1b. 26 chữ cái + set | `O(n)` (dựng set) | `O(n)` (tối đa 26 phần tử khác nhau → thực chất `O(1)`) |
| 2. Binary search tự viết | `O(log n)` | `O(1)` |
| 3. `bisect_right` | `O(log n)` | `O(1)` |

Với `n = 10^4`: tuyến tính tối đa 10.000 phép so sánh; binary search chỉ khoảng `log2(10^4) ≈ 13.3`, tức 13–14 vòng. Cell dưới đếm số vòng thật sự (trường hợp mọi phần tử đều False).

In [15]:
import math

def count_iterations(n):
    letters = ["a"] * (n - 1) + ["b"]
    lo, hi, it = 0, n, 0
    while lo < hi:
        it += 1
        mid = (lo + hi) // 2
        if letters[mid] > "z":
            hi = mid
        else:
            lo = mid + 1
    return it

for n in [2, 10, 100, 10_000]:
    print(f"n = {n:>6}: binary search chạy {count_iterations(n):>2} vòng | log2(n) ≈ {math.log2(n):.1f}")

n =      2: binary search chạy  1 vòng | log2(n) ≈ 1.0
n =     10: binary search chạy  3 vòng | log2(n) ≈ 3.3
n =    100: binary search chạy  6 vòng | log2(n) ≈ 6.6
n =  10000: binary search chạy 13 vòng | log2(n) ≈ 13.3


## 11. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual!r}")

sol = Solution()
check("đề 1", sol.nextGreatestLetter(["c", "f", "j"], "a"), "c")
check("đề 2 (bằng target)", sol.nextGreatestLetter(["c", "f", "j"], "c"), "f")
check("đề 3 (quay vòng)", sol.nextGreatestLetter(["x", "x", "y", "y"], "z"), "x")
check("target = phần tử cuối", sol.nextGreatestLetter(["c", "f", "j"], "j"), "c")
check("target giữa hai phần tử", sol.nextGreatestLetter(["c", "f", "j"], "d"), "f")
check("nhiều phần tử bằng target", sol.nextGreatestLetter(["e", "e", "e", "k", "q", "q"], "e"), "k")
check("hai phần tử", sol.nextGreatestLetter(["a", "b"], "a"), "b")
check("hai phần tử, quay vòng", sol.nextGreatestLetter(["a", "b"], "b"), "a")
check("target nhỏ hơn tất cả", sol.nextGreatestLetter(["m", "n"], "a"), "m")

PASS  đề 1: 'c'
PASS  đề 2 (bằng target): 'f'
PASS  đề 3 (quay vòng): 'x'
PASS  target = phần tử cuối: 'c'
PASS  target giữa hai phần tử: 'f'
PASS  nhiều phần tử bằng target: 'k'
PASS  hai phần tử: 'b'
PASS  hai phần tử, quay vòng: 'a'
PASS  target nhỏ hơn tất cả: 'm'


In [17]:
import random
import string

rng = random.Random(744)
impls = [next_letter_linear, next_letter_alphabet, next_letter_binary, next_letter_bisect,
         Solution().nextGreatestLetter, SolutionBisect().nextGreatestLetter, SolutionLinear().nextGreatestLetter]
for _ in range(5000):
    n = rng.randint(2, 12)
    while True:
        letters = sorted(rng.choice("abcdefghij") for _ in range(n))
        if len(set(letters)) >= 2:          # đúng constraint: >= 2 ký tự khác nhau
            break
    target = rng.choice(string.ascii_lowercase[:12])
    outs = {f(letters, target) for f in impls}
    assert len(outs) == 1, (letters, target, outs)
print("5000 bộ ngẫu nhiên: 7 cài đặt khớp nhau")

5000 bộ ngẫu nhiên: 7 cài đặt khớp nhau


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `>=` thay vì `>` (hay `bisect_left` thay vì `bisect_right`)

In [18]:
def wrong_ge(letters, target):
    i = bisect_left(letters, target)            # tìm phần tử >= target
    return letters[i % len(letters)]

print("sai :", wrong_ge(["c", "f", "j"], "c"), "| đúng:", next_letter_bisect(["c", "f", "j"], "c"))

sai : c | đúng: f


### Lỗi 2: Quên quay vòng → `IndexError`

Khi mọi ký tự `<= target`, `lo = n` → `letters[n]` vượt chỉ số.

In [19]:
def wrong_no_wrap(letters, target):
    lo, hi = 0, len(letters)
    while lo < hi:
        mid = (lo + hi) // 2
        if letters[mid] > target:
            hi = mid
        else:
            lo = mid + 1
    return letters[lo]                          # thiếu % n

try:
    wrong_no_wrap(["x", "x", "y", "y"], "z")
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Sửa:** `letters[lo % len(letters)]`, hoặc `if lo == n: return letters[0]`.

### Lỗi 3: Trộn hai kiểu biên — `hi = n - 1` với `while lo < hi`

Kiểu nửa khoảng `[lo, hi)` cần `hi = n` để biểu diễn "không có đáp án". Nếu khởi tạo `hi = n - 1`, kết quả không bao giờ đạt `n` → trường hợp quay vòng trả sai ký tự cuối.

In [20]:
def wrong_hi_init(letters, target):
    lo, hi = 0, len(letters) - 1                # SAI: không biểu diễn được "không có"
    while lo < hi:
        mid = (lo + hi) // 2
        if letters[mid] > target:
            hi = mid
        else:
            lo = mid + 1
    return letters[lo % len(letters)]

print("sai :", wrong_hi_init(["x", "x", "y", "y"], "z"), "| đúng:", next_letter_binary(["x", "x", "y", "y"], "z"))

sai : y | đúng: x


### Lỗi 4: `lo = mid` thay vì `lo = mid + 1` → lặp vô hạn

Khi `hi = lo + 1`, `mid = lo`; nếu nhánh False gán `lo = mid` thì `lo` không đổi → kẹt mãi. Cell dưới giới hạn số vòng để minh họa an toàn.

In [21]:
def wrong_lo_mid(letters, target, max_iter=30):
    lo, hi, it = 0, len(letters), 0
    while lo < hi:
        it += 1
        if it > max_iter:
            return f"kẹt sau {max_iter} vòng (lo={lo}, hi={hi})"
        mid = (lo + hi) // 2
        if letters[mid] > target:
            hi = mid
        else:
            lo = mid                            # SAI
    return letters[lo % len(letters)]

print(wrong_lo_mid(["c", "f", "j"], "z"))

kẹt sau 30 vòng (lo=2, hi=3)


### Lỗi 5: Trả `min` của các ký tự lớn hơn mà quên trường hợp rỗng

`min(ch for ch in letters if ch > target)` ném `ValueError` khi không có ký tự nào lớn hơn. Dùng `min(..., default=letters[0])`.

In [22]:
try:
    min(ch for ch in ["x", "y"] if ch > "z")
except ValueError as e:
    print("ValueError:", e)
print(min((ch for ch in ["x", "y"] if ch > "z"), default="x"))

ValueError: min() iterable argument is empty
x


## 13. Mẹo nhận dạng pattern

- Đề cho mảng **đã sắp xếp** + hỏi "nhỏ nhất lớn hơn", "lớn nhất nhỏ hơn", "vị trí chèn", "đầu tiên/cuối cùng ..." → **Binary Search**.
- Viết điều kiện thành dãy đơn điệu `F F F T T T` và tìm **True đầu tiên** — khung chung:

```python
lo, hi = 0, n
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid
    else:      lo = mid + 1
# lo = vị trí True đầu tiên (n nếu không có)
```

- `P(i) = a[i] >= x` → lower bound (`bisect_left`); `P(i) = a[i] > x` → upper bound (`bisect_right`).
- "Quay vòng / vòng tròn" → `% n`.
- Bài cùng họ: 35 Search Insert Position, 34 Find First and Last Position, 278 First Bad Version, 704 Binary Search, 2529 Maximum Count of Positive and Negative Integer.

## 14. Biến thể

### 14.1. Ký tự **lớn nhất nhỏ hơn hẳn** target (quay vòng về cuối)

Vị trí True đầu tiên của `a[i] >= target` là `bisect_left`; phần tử ngay **trước** nó là phần tử lớn nhất `< target`. Nếu chỉ số là `-1` thì Python tự quay về **cuối** mảng nhờ chỉ số âm!

In [23]:
def prev_smaller_letter(letters, target):
    i = bisect_left(letters, target) - 1
    return letters[i]            # i = -1 -> letters[-1]: quay vòng về cuối

print(prev_smaller_letter(["c", "f", "j"], "g"))   # f
print(prev_smaller_letter(["c", "f", "j"], "c"))   # j (không có ký tự < c -> quay vòng)

f
j


### 14.2. Đếm số phần tử lớn hơn target

`n - bisect_right(a, target)` — vì mọi phần tử từ vị trí `bisect_right` trở đi đều lớn hơn.

In [24]:
a = ["c", "c", "f", "f", "j", "m"]
for t in ["a", "c", "f", "m"]:
    print(f"số phần tử > {t!r}: {len(a) - bisect_right(a, t)}")

số phần tử > 'a': 6
số phần tử > 'c': 4
số phần tử > 'f': 2
số phần tử > 'm': 0


### 14.3. LeetCode 34 — vị trí đầu và cuối của target

Đầu = `bisect_left(a, x)`; cuối = `bisect_right(a, x) - 1`. Nếu đầu ra ngoài hoặc `a[đầu] != x` → không có.

In [25]:
def search_range(nums, x):
    lo = bisect_left(nums, x)
    if lo == len(nums) or nums[lo] != x:
        return [-1, -1]
    return [lo, bisect_right(nums, x) - 1]

print(search_range([5, 7, 7, 8, 8, 10], 8))   # [3, 4]
print(search_range([5, 7, 7, 8, 8, 10], 6))   # [-1, -1]

[3, 4]
[-1, -1]


## 15. Bài tập

### Bài 1 — Trace bằng tay
Với `letters = ["a","c","c","e","g","g","k"]` và `target = "c"`, lập bảng `lo, hi, mid, letters[mid]` theo Cách 2. Đáp án là gì?

### Bài 2 — Viết `lower_bound`
Trả vị trí đầu tiên `i` có `a[i] >= x` (hoặc `len(a)`). Không dùng `bisect`.

In [26]:
def lower_bound(a: list, x) -> int:
    # TODO: khung "True đầu tiên" với P(i) = a[i] >= x
    pass

# assert lower_bound([1, 3, 3, 5], 3) == 1
# assert lower_bound([1, 3, 3, 5], 4) == 3
# assert lower_bound([1, 3, 3, 5], 9) == 4

### Bài 3 — Điền chỗ trống lời giải chính

In [27]:
def exercise_next_letter(letters: list[str], target: str) -> str:
    lo, hi = 0, len(letters)
    while lo < hi:
        mid = (lo + hi) // 2
        # TODO 1: nếu letters[mid] > target -> thu hẹp bên phải (giữ mid)
        # TODO 2: ngược lại -> bỏ mid và bên trái
        pass
    # TODO 3: trả kết quả có quay vòng
    pass

# assert exercise_next_letter(["c", "f", "j"], "c") == "f"
# assert exercise_next_letter(["x", "x", "y", "y"], "z") == "x"

### Bài 4 — Search Insert Position (LeetCode 35)
Cho mảng số **tăng dần, không trùng** `nums` và `target`. Trả chỉ số của `target` nếu có, nếu không thì vị trí cần chèn để giữ thứ tự.

In [28]:
def search_insert(nums: list[int], target: int) -> int:
    # TODO
    pass

# assert search_insert([1, 3, 5, 6], 5) == 2
# assert search_insert([1, 3, 5, 6], 2) == 1
# assert search_insert([1, 3, 5, 6], 7) == 4

### Bài 5 — Đếm ký tự nằm trong khoảng
Đếm số phần tử `ch` của `letters` (đã sắp xếp) thỏa `lo_ch <= ch <= hi_ch`, trong `O(log n)`.

In [29]:
def count_in_range(letters: list[str], lo_ch: str, hi_ch: str) -> int:
    # TODO
    pass

# assert count_in_range(["a", "c", "c", "e", "g", "g", "k"], "c", "g") == 5
# assert count_in_range(["a", "c", "c", "e"], "x", "z") == 0

### Bài 6 — Mini challenge: First Bad Version (LeetCode 278)
Các phiên bản `1..n`; từ một phiên bản nào đó trở đi đều "hỏng". Cho hàm `is_bad(v)`, tìm phiên bản hỏng **đầu tiên** với số lần gọi `is_bad` là `O(log n)`.

In [30]:
def first_bad_version(n: int, is_bad) -> int:
    # TODO
    pass

# assert first_bad_version(5, lambda v: v >= 4) == 4
# assert first_bad_version(1, lambda v: v >= 1) == 1
# assert first_bad_version(2**31 - 1, lambda v: v >= 1_234_567) == 1_234_567

`lambda v: v >= 4` là **hàm ẩn danh** một dòng: nhận `v`, trả `v >= 4`. Tiện để truyền một hàm nhỏ làm tham số.

## 16. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Viết dãy `F/T` cho `> 'c'` trước | `lo=0, hi=7` | Vòng 1: `mid=3 ('e')` → T → `hi=3` ... |
| 2 | Giống Cách 2, đổi điều kiện | `P(mid) = a[mid] >= x` | `if a[mid] >= x: hi = mid else: lo = mid + 1` |
| 3 | Đúng khung True đầu tiên | `hi = mid` / `lo = mid + 1` | `return letters[lo % len(letters)]` |
| 4 | Vị trí chèn = phần tử đầu tiên `>= target` | Chính là lower bound | `return lower_bound(nums, target)` |
| 5 | Hiệu hai vị trí | trái = đầu tiên `>= lo_ch`; phải = đầu tiên `> hi_ch` | `bisect_right(a, hi_ch) - bisect_left(a, lo_ch)` |
| 6 | Dãy F...F T...T trên `1..n` | `lo = 1, hi = n` (đáp án chắc chắn tồn tại) | `if is_bad(mid): hi = mid else: lo = mid + 1` |

## 17. Đáp án tham khảo

**Bài 1:** dãy `> 'c'`: `a c c e g g k` → `F F F T T T T`, đáp án vị trí 3 (`'e'`).

| Vòng | lo | hi | mid | letters[mid] | > 'c'? | Cập nhật |
|---:|---:|---:|---:|:---:|:---:|---|
| 1 | 0 | 7 | 3 | e | ✅ | hi = 3 |
| 2 | 0 | 3 | 1 | c | ❌ | lo = 2 |
| 3 | 2 | 3 | 2 | c | ❌ | lo = 3 |
| — | 3 | 3 | | | | trả `'e'` |

In [31]:
trace_next_letter(["a", "c", "c", "e", "g", "g", "k"], "c")


def answer_lower_bound(a, x):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] >= x:
            hi = mid
        else:
            lo = mid + 1
    return lo


def answer_next_letter(letters, target):
    lo, hi = 0, len(letters)
    while lo < hi:
        mid = (lo + hi) // 2
        if letters[mid] > target:
            hi = mid
        else:
            lo = mid + 1
    return letters[lo % len(letters)]


def answer_search_insert(nums, target):
    return answer_lower_bound(nums, target)


def answer_count_in_range(letters, lo_ch, hi_ch):
    return max(0, bisect_right(letters, hi_ch) - bisect_left(letters, lo_ch))


def answer_first_bad_version(n, is_bad):
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi) // 2
        if is_bad(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


assert answer_lower_bound([1, 3, 3, 5], 3) == 1
assert answer_lower_bound([1, 3, 3, 5], 4) == 3
assert answer_lower_bound([1, 3, 3, 5], 9) == 4
assert answer_next_letter(["c", "f", "j"], "c") == "f"
assert answer_next_letter(["x", "x", "y", "y"], "z") == "x"
assert answer_search_insert([1, 3, 5, 6], 5) == 2
assert answer_search_insert([1, 3, 5, 6], 2) == 1
assert answer_search_insert([1, 3, 5, 6], 7) == 4
assert answer_count_in_range(["a", "c", "c", "e", "g", "g", "k"], "c", "g") == 5
assert answer_count_in_range(["a", "c", "c", "e"], "x", "z") == 0
assert answer_first_bad_version(5, lambda v: v >= 4) == 4
assert answer_first_bad_version(1, lambda v: v >= 1) == 1
assert answer_first_bad_version(2**31 - 1, lambda v: v >= 1_234_567) == 1_234_567
print("Tất cả đáp án PASS")

letters = ['a', 'c', 'c', 'e', 'g', 'g', 'k'], target = 'c'
vòng 1: lo=0, hi=7, mid=3, letters[mid]='e', > target? True  -> lo=0, hi=3
vòng 2: lo=0, hi=3, mid=1, letters[mid]='c', > target? False -> lo=2, hi=3
vòng 3: lo=2, hi=3, mid=2, letters[mid]='c', > target? False -> lo=3, hi=3
dừng: lo = hi = 3 -> trả letters[3 % 7] = 'e'
Tất cả đáp án PASS


Ghi chú bài 5: `max(0, ...)` phòng trường hợp `lo_ch > hi_ch` (khoảng rỗng) làm hiệu âm. Bài 6 với `n = 2^31 - 1` chỉ gọi `is_bad` khoảng 31 lần — sức mạnh của `O(log n)`.

## 18. Template nhớ nhanh + Checklist submit

```python
class Solution:
    def nextGreatestLetter(self, letters, target):
        lo, hi = 0, len(letters)              # nửa khoảng [lo, hi), hi = n nghĩa là "không có"
        while lo < hi:
            mid = (lo + hi) // 2
            if letters[mid] > target:         # LỚN HƠN HẲN
                hi = mid                      # mid có thể là đáp án -> giữ
            else:
                lo = mid + 1                  # mid chắc chắn không phải -> bỏ
        return letters[lo % len(letters)]     # quay vòng khi lo == n
```

**Câu thần chú:** *"dãy F…T — tìm True đầu — hi = mid, lo = mid + 1 — % n để quay vòng."*

**Checklist trước khi submit:**
- [ ] Điều kiện là `>` (lớn hơn hẳn), không phải `>=`?
- [ ] `hi` khởi tạo bằng `len(letters)` (không phải `len - 1`) khi dùng `while lo < hi`?
- [ ] Nhánh False là `lo = mid + 1` (tránh kẹt vòng lặp)?
- [ ] Có `% len(letters)` hoặc xử lý `lo == n` → `letters[0]`?
- [ ] Đã test `target` bằng một phần tử có lặp, `target` ≥ phần tử cuối, `target` < phần tử đầu?